# Module 5: Probability Theory

Probability is the **language of uncertainty**, and AI is fundamentally about making decisions under uncertainty.

### 🎯 What you'll learn:
- Axioms of probability
- Conditional probability and Bayes' theorem
- Random variables (discrete and continuous)
- PMFs, PDFs, CDFs
- Expectation, variance, covariance
- Law of Large Numbers and Central Limit Theorem

### 🤖 Why it matters for AI:
- **Bayesian inference** powers many ML models
- **Generative models** (VAEs, diffusion) are all about probability distributions
- **Classification** outputs are probabilities
- **Reinforcement learning** uses expected rewards

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from collections import Counter

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.style.use('seaborn-v0_8-darkgrid')

---
## 1. Axioms of Probability

### Kolmogorov Axioms:
1. $P(A) \geq 0$ for any event $A$
2. $P(\Omega) = 1$ (something must happen)
3. For mutually exclusive events: $P(A \cup B) = P(A) + P(B)$

### Derived Rules:
- **Complement**: $P(A^c) = 1 - P(A)$
- **Union**: $P(A \cup B) = P(A) + P(B) - P(A \cap B)$
- **Total Probability**: $P(A) = \sum_i P(A|B_i)P(B_i)$

In [ ]:
# Simulate coin flips
n_flips = 10000
flips = np.random.choice(['H', 'T'], size=n_flips)
counts = Counter(flips)
print(f"Coin flip simulation ({n_flips} flips):")
print(f"  P(H) ≈ {counts['H']/n_flips:.4f} (expected: 0.5)")
print(f"  P(T) ≈ {counts['T']/n_flips:.4f}")
print(f"  P(H) + P(T) = {(counts['H']+counts['T'])/n_flips:.4f} (axiom 2)")

# Dice example
n_rolls = 100000
rolls = np.random.randint(1, 7, size=n_rolls)
print(f"\nDice simulation ({n_rolls} rolls):")
for i in range(1, 7):
    print(f"  P({i}) ≈ {np.mean(rolls == i):.4f}")

# P(even) = P(2) + P(4) + P(6) (mutually exclusive)
p_even = np.mean((rolls == 2) | (rolls == 4) | (rolls == 6))
print(f"  P(even) ≈ {p_even:.4f} (expected: 0.5)")

---
## 2. Conditional Probability and Independence

### Conditional Probability
$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

"The probability of $A$ given that $B$ has occurred."

### Independence
$A$ and $B$ are **independent** if:
$$P(A \cap B) = P(A) \cdot P(B)$$

Equivalently: $P(A|B) = P(A)$ — knowing $B$ doesn't change $A$'s probability.

### 🤖 AI Connection:
- **Naive Bayes** classifier assumes feature independence
- **Graphical models** encode conditional independence

In [ ]:
# Conditional probability: Two dice
n = 100000
die1 = np.random.randint(1, 7, n)
die2 = np.random.randint(1, 7, n)
total = die1 + die2

# P(die1=6 | total=8)
mask_8 = total == 8
p_6_given_8 = np.mean(die1[mask_8] == 6)
print(f"P(die1=6 | total=8) ≈ {p_6_given_8:.4f}")
print(f"Exact: P = 1/5 = {1/5:.4f}")
# Ways to get 8: (2,6),(3,5),(4,4),(5,3),(6,2) → 1 out of 5 has die1=6

# Independence check
A = die1 >= 4  # Event A: die1 ≥ 4
B = die2 <= 3  # Event B: die2 ≤ 3
print(f"\nIndependence check:")
print(f"  P(A) = {np.mean(A):.4f}")
print(f"  P(B) = {np.mean(B):.4f}")
print(f"  P(A∩B) = {np.mean(A & B):.4f}")
print(f"  P(A)·P(B) = {np.mean(A)*np.mean(B):.4f}")
print(f"  Independent? {np.isclose(np.mean(A & B), np.mean(A)*np.mean(B), atol=0.01)}")

---
## 3. Bayes' Theorem — The Foundation of Bayesian ML

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

In ML language:
$$\underbrace{P(\theta|D)}_{\text{posterior}} = \frac{\underbrace{P(D|\theta)}_{\text{likelihood}} \cdot \underbrace{P(\theta)}_{\text{prior}}}{\underbrace{P(D)}_{\text{evidence}}}$$

- **Prior** $P(\theta)$: What we believe before seeing data
- **Likelihood** $P(D|\theta)$: How likely the data is given parameters
- **Posterior** $P(\theta|D)$: Updated beliefs after seeing data
- **Evidence** $P(D)$: Normalizing constant

### 🤖 AI Connection:
- **Bayesian neural networks** maintain distributions over weights
- **Spam filters** use Naive Bayes
- **Medical diagnosis** AI uses Bayes' theorem

In [ ]:
# Classic Bayes' Theorem: Medical Test
# Disease prevalence: 1%
# Test sensitivity (true positive rate): 99%
# Test specificity (true negative rate): 95%

P_disease = 0.01
P_positive_given_disease = 0.99    # Sensitivity
P_negative_given_healthy = 0.95    # Specificity
P_positive_given_healthy = 1 - P_negative_given_healthy  # False positive = 5%

# P(disease | positive test) = ?
P_positive = P_positive_given_disease * P_disease + P_positive_given_healthy * (1 - P_disease)
P_disease_given_positive = (P_positive_given_disease * P_disease) / P_positive

print("=== Medical Test (Bayes' Theorem) ===")
print(f"P(disease) = {P_disease}")
print(f"P(+|disease) = {P_positive_given_disease}")
print(f"P(+|healthy) = {P_positive_given_healthy}")
print(f"\nP(disease|+) = {P_disease_given_positive:.4f}")
print(f"\n⚠️ Even with a 99% accurate test, a positive result")
print(f"only means ~{P_disease_given_positive*100:.1f}% chance of disease!")
print(f"(Because the disease is rare, most positives are false positives)")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
labels = ['True Positive\n(disease + positive)', 'False Positive\n(healthy + positive)',
          'False Negative\n(disease + negative)', 'True Negative\n(healthy + negative)']
sizes = [P_positive_given_disease * P_disease,
         P_positive_given_healthy * (1-P_disease),
         (1-P_positive_given_disease) * P_disease,
         P_negative_given_healthy * (1-P_disease)]
colors = ['#E74C3C', '#F39C12', '#3498DB', '#2ECC71']
ax.barh(labels, sizes, color=colors)
ax.set_xlabel('Probability', fontsize=14)
ax.set_title('Bayes\' Theorem: Medical Test Breakdown', fontsize=16)
for i, v in enumerate(sizes):
    ax.text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. Random Variables, PMFs, PDFs, CDFs

### Discrete Random Variable
- **PMF** (Probability Mass Function): $P(X = x)$
- **CDF** (Cumulative Distribution Function): $F(x) = P(X \leq x)$

### Continuous Random Variable
- **PDF** (Probability Density Function): $f(x)$ where $P(a \leq X \leq b) = \int_a^b f(x)dx$
- **CDF**: $F(x) = P(X \leq x) = \int_{-\infty}^{x} f(t)dt$

Key properties:
- $\int_{-\infty}^{\infty} f(x)dx = 1$ (total probability = 1)
- $f(x) \geq 0$ (no negative probabilities)
- $F'(x) = f(x)$ (PDF is derivative of CDF)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Discrete: Binomial(n=10, p=0.3)
n, p = 10, 0.3
k = np.arange(0, 11)
pmf = stats.binom.pmf(k, n, p)
cdf = stats.binom.cdf(k, n, p)

axes[0, 0].bar(k, pmf, color='#3498DB', alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Binomial PMF (n=10, p=0.3)', fontsize=12)
axes[0, 0].set_xlabel('k'); axes[0, 0].set_ylabel('P(X=k)')

axes[0, 1].step(k, cdf, where='mid', color='#E74C3C', linewidth=2)
axes[0, 1].set_title('Binomial CDF', fontsize=12)

# Samples
samples = np.random.binomial(n, p, 5000)
axes[0, 2].hist(samples, bins=np.arange(-0.5, 11.5, 1), density=True, alpha=0.7, color='#2ECC71', edgecolor='black')
axes[0, 2].bar(k, pmf, alpha=0.5, color='red', width=0.3, label='True PMF')
axes[0, 2].set_title('Samples vs PMF', fontsize=12); axes[0, 2].legend()

# Continuous: Normal(μ=0, σ=1)
x_vals = np.linspace(-4, 4, 200)
pdf = stats.norm.pdf(x_vals, 0, 1)
cdf_vals = stats.norm.cdf(x_vals, 0, 1)

axes[1, 0].plot(x_vals, pdf, 'b-', linewidth=2)
axes[1, 0].fill_between(x_vals, pdf, alpha=0.2, color='#3498DB')
axes[1, 0].set_title('Normal PDF (μ=0, σ=1)', fontsize=12)

axes[1, 1].plot(x_vals, cdf_vals, 'r-', linewidth=2)
axes[1, 1].set_title('Normal CDF', fontsize=12)

samples_norm = np.random.randn(5000)
axes[1, 2].hist(samples_norm, bins=50, density=True, alpha=0.7, color='#2ECC71')
axes[1, 2].plot(x_vals, pdf, 'r-', linewidth=2, label='True PDF')
axes[1, 2].set_title('Samples vs PDF', fontsize=12); axes[1, 2].legend()

plt.suptitle('Discrete vs Continuous Random Variables', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Expectation, Variance, and Covariance

### Expectation (Mean)
$$E[X] = \sum_x x \cdot P(X=x) \quad \text{or} \quad E[X] = \int_{-\infty}^{\infty} x \cdot f(x) \, dx$$

### Variance
$$\text{Var}(X) = E[(X - \mu)^2] = E[X^2] - (E[X])^2$$

### Covariance
$$\text{Cov}(X, Y) = E[(X - \mu_X)(Y - \mu_Y)] = E[XY] - E[X]E[Y]$$

### Properties:
- $E[aX + b] = aE[X] + b$ (linearity)
- $\text{Var}(aX + b) = a^2 \text{Var}(X)$
- If $X, Y$ independent: $\text{Var}(X+Y) = \text{Var}(X) + \text{Var}(Y)$

### 🤖 AI Connection:
- **Loss functions** compute expected loss
- **Batch normalization** uses running mean and variance
- **Covariance matrices** describe feature relationships

In [ ]:
# Expectation and Variance
# Discrete: fair die
die_values = np.array([1, 2, 3, 4, 5, 6])
die_probs = np.ones(6) / 6

E_X = np.sum(die_values * die_probs)
E_X2 = np.sum(die_values**2 * die_probs)
Var_X = E_X2 - E_X**2

print(f"Fair Die:")
print(f"  E[X] = {E_X:.4f} (expected: 3.5)")
print(f"  Var(X) = {Var_X:.4f} (expected: 35/12 ≈ {35/12:.4f})")
print(f"  σ(X) = {np.sqrt(Var_X):.4f}")

# Verify with simulation
samples = np.random.randint(1, 7, 100000)
print(f"\n  Simulated E[X] = {samples.mean():.4f}")
print(f"  Simulated Var(X) = {samples.var():.4f}")

# Covariance example
np.random.seed(42)
X = np.random.randn(10000)
Y = 2 * X + np.random.randn(10000)  # Y is correlated with X
Z = np.random.randn(10000)  # Z is independent

print(f"\nCovariance:")
print(f"  Cov(X, Y) = {np.cov(X, Y)[0, 1]:.4f} (positive — correlated)")
print(f"  Cov(X, Z) = {np.cov(X, Z)[0, 1]:.4f} (≈ 0 — independent)")
print(f"  Correlation(X, Y) = {np.corrcoef(X, Y)[0, 1]:.4f}")

---
## 6. Joint and Marginal Distributions

### Joint Distribution
$P(X = x, Y = y)$ or $f_{X,Y}(x, y)$

### Marginal Distribution
$$P(X = x) = \sum_y P(X = x, Y = y)$$
$$f_X(x) = \int_{-\infty}^{\infty} f_{X,Y}(x, y) \, dy$$

### 🤖 AI Connection:
- **Marginalization** removes unwanted variables
- VAEs use marginal likelihood: $P(x) = \int P(x|z) P(z) dz$

In [ ]:
# Joint distribution example: Bivariate normal
mean = [1, 2]
cov = [[1, 0.8], [0.8, 1.5]]
samples = np.random.multivariate_normal(mean, cov, 5000)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Joint distribution
axes[0].scatter(samples[:, 0], samples[:, 1], alpha=0.1, s=5, color='#3498DB')
axes[0].set_title('Joint Distribution P(X,Y)', fontsize=14)
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')

# Marginal X (integrate out Y)
axes[1].hist(samples[:, 0], bins=50, density=True, alpha=0.7, color='#E74C3C')
x_range = np.linspace(-3, 5, 100)
axes[1].plot(x_range, stats.norm.pdf(x_range, mean[0], np.sqrt(cov[0][0])), 'k-', lw=2)
axes[1].set_title('Marginal P(X) = ∫P(X,Y)dY', fontsize=14)

# Marginal Y
axes[2].hist(samples[:, 1], bins=50, density=True, alpha=0.7, color='#2ECC71')
y_range = np.linspace(-2, 6, 100)
axes[2].plot(y_range, stats.norm.pdf(y_range, mean[1], np.sqrt(cov[1][1])), 'k-', lw=2)
axes[2].set_title('Marginal P(Y) = ∫P(X,Y)dX', fontsize=14)

plt.tight_layout()
plt.show()

---
## 7. Law of Large Numbers

As $n \to \infty$, the sample mean converges to the true mean:

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i \xrightarrow{n \to \infty} \mu$$

### 🤖 AI Connection:
- SGD works because mini-batch gradients approximate true gradients (by LLN)
- Monte Carlo methods rely on LLN for convergence

In [ ]:
# Law of Large Numbers visualization
np.random.seed(42)
true_mean = 3.5  # Fair die

samples = np.random.randint(1, 7, 10000)
running_mean = np.cumsum(samples) / np.arange(1, len(samples) + 1)

plt.figure(figsize=(12, 5))
plt.plot(running_mean, color='#3498DB', linewidth=1)
plt.axhline(y=true_mean, color='#E74C3C', linestyle='--', linewidth=2, label=f'True mean = {true_mean}')
plt.xlabel('Number of samples', fontsize=14)
plt.ylabel('Running average', fontsize=14)
plt.title('Law of Large Numbers: Sample Mean → True Mean', fontsize=16)
plt.legend(fontsize=12)
plt.xscale('log')
plt.grid(True, alpha=0.3)
plt.show()

---
## 8. Central Limit Theorem (CLT)

The sum/mean of many independent random variables is approximately **normally distributed**, regardless of the original distribution:

$$\frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \xrightarrow{d} N(0, 1)$$

### Why is CLT so important?
- It explains why the normal distribution appears everywhere
- It justifies using Gaussian assumptions in many models
- It tells us how fast averages converge ($O(1/\sqrt{n})$)

In [ ]:
# CLT: Sum of uniform random variables → Normal
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, n in enumerate([1, 2, 5, 10, 30, 100]):
    ax = axes[idx // 3, idx % 3]
    
    # Generate 10000 sample means, each from n uniform[0,1] samples
    sample_means = np.mean(np.random.uniform(0, 1, (10000, n)), axis=1)
    
    ax.hist(sample_means, bins=50, density=True, alpha=0.7, color='#3498DB', edgecolor='white')
    
    # Overlay normal approximation
    mu = 0.5  # True mean of Uniform(0,1)
    sigma = 1/np.sqrt(12)  # True std of Uniform(0,1)
    x_range = np.linspace(sample_means.min(), sample_means.max(), 100)
    ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma/np.sqrt(n)), 'r-', lw=2, label='Normal approx')
    
    ax.set_title(f'n = {n}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Central Limit Theorem: Mean of n Uniform Samples → Normal', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Correlation vs Causation

### Correlation
$$\rho_{X,Y} = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y} \in [-1, 1]$$

### Key Facts:
- $\rho = 1$: perfect positive linear relationship
- $\rho = -1$: perfect negative linear relationship
- $\rho = 0$: no linear relationship (but could still be dependent!)
- **Correlation ≠ Causation**: Ice cream sales and drowning are correlated (both increase in summer)

In [ ]:
# Different correlation patterns
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

n = 500
# Strong positive
X = np.random.randn(n)
Y_pos = X + 0.2*np.random.randn(n)
axes[0].scatter(X, Y_pos, s=10, alpha=0.5)
axes[0].set_title(f'ρ = {np.corrcoef(X, Y_pos)[0,1]:.2f}', fontsize=14)

# Negative
Y_neg = -X + 0.3*np.random.randn(n)
axes[1].scatter(X, Y_neg, s=10, alpha=0.5, color='red')
axes[1].set_title(f'ρ = {np.corrcoef(X, Y_neg)[0,1]:.2f}', fontsize=14)

# No correlation
Y_none = np.random.randn(n)
axes[2].scatter(X, Y_none, s=10, alpha=0.5, color='green')
axes[2].set_title(f'ρ = {np.corrcoef(X, Y_none)[0,1]:.2f}', fontsize=14)

# Non-linear (corr=0 but dependent!)
Y_circle = X**2 + 0.1*np.random.randn(n)
axes[3].scatter(X, Y_circle, s=10, alpha=0.5, color='purple')
axes[3].set_title(f'ρ = {np.corrcoef(X, Y_circle)[0,1]:.2f}\n(but dependent!)', fontsize=12)

plt.suptitle('Correlation Patterns', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

---
## 10. Summary: Probability for AI

| Concept | AI Application |
|---------|---------------|
| Bayes' Theorem | Bayesian inference, posterior estimation |
| Conditional Probability | Classification, filtering |
| Expectation | Loss functions, reward signals |
| Variance | Uncertainty quantification |
| LLN | Mini-batch SGD convergence |
| CLT | Why Gaussian assumptions work |
| Covariance | Feature relationships, PCA |

**Next: Specific probability distributions and statistical inference!** 🚀